# Sync MongoDB => DeltaLake

Descarga OTs que no esten presentes en el DeltaLake desde MongoDB

In [5]:
# Ubicación del Delta Talbe

DELTA_PATH = "/home/vlad/delta_v23"

### Carga librerias

In [2]:
from importlib import reload
from eerssa import gestionOT as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import matrizActividades as Actividades     # process ot.data["actividades"]

In [3]:
reload( OrdenTrabajo )
reload( Actividades  )

<module 'eerssa.matrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/matrizActividades.py'>

In [1]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import re
import json
import pickle
import logging
import pymongo
from pymongo.errors import ConnectionFailure
from deltalake import DeltaTable, write_deltalake
from pprint import pprint
from datetime import datetime

from eerssa.secret import Keys

logging.basicConfig(level=logging.INFO)

# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.

Success!!!


INFO:root::::: Conexion exitosa con MongoDB ::::


### Descargar ot_id desde Mongo

In [6]:
""" 
   CARGAR LA BASE DE DATOS DESDE DELTA LAKE
"""

if not DeltaTable.is_deltatable(DELTA_PATH):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {DELTA_PATH}" )
else:
    dt = DeltaTable(DELTA_PATH)
    df = dt.to_pandas()
    print(f"Conectado a la tabla Delta Lake en: {DELTA_PATH}")

Conectado a la tabla Delta Lake en: /home/vlad/delta_v23


In [11]:
""" 
   OBTENER TODOS LOS 'id_ot' desde MongoDB
"""

try:
    # 1. Use a projection to only retrieve the 'id_ot' field.
    #    - {'id_ot': 1} means "include this field".
    #    - {'_id': 0} means "exclude the default _id field".
    cursor = CurrentCollection.find({}, {'id_ot': 1, '_id': 0})

    # 2. Create a list from the cursor results using a list comprehension.
    #    This iterates through each document in the cursor and extracts 'id_ot'.
    id_ot_list = [doc['id_ot'] for doc in cursor]

    # 3. Now you have your list of all 'id_ot' values.
    print(f"Successfully retrieved {len(id_ot_list)} 'id_ot' values.")
    if id_ot_list:
        print("First 10 values:", id_ot_list[:10])

except Exception as e:
    print(f"An error occurred: {e}")


Successfully retrieved 21941 'id_ot' values.
First 10 values: [148859, 150217, 148664, 148843, 149271, 149150, 155468, 155428, 155505, 155585]


In [16]:
"""
   Obtener todos los 'id_ot' existentes en DeltaLake
"""

delta_ids = df["id_ot"].unique()
len(delta_ids)

21879

In [17]:
"""
   Difentecia de las ot que faltan en DeltaLake
"""
set_mongo = set(id_ot_list)
set_delta = set(delta_ids)

# Find which items in set_delta are not in set_mongo
new_ids_set = set_mongo.difference(set_delta)

# Convert the result back to a list
new_ids_to_process = list(new_ids_set)

print(f"Found {len(new_ids_to_process)} new IDs to be processed.")
# We sort the list here just for a predictable, clean output
print(f"New IDs: {sorted(new_ids_to_process)}")

Found 62 new IDs to be processed.
New IDs: [157262, 157480, 157541, 157561, 157883, 157930, 157986, 158000, 158003, 158006, 158010, 158014, 158016, 158019, 158020, 158024, 158028, 158090, 158095, 158097, 158099, 158101, 158103, 158109, 158112, 158113, 158119, 158127, 158130, 158135, 158144, 158174, 158181, 158192, 158193, 158199, 158202, 158206, 158207, 158208, 158209, 158210, 158211, 158212, 158224, 158252, 158260, 158264, 158265, 158271, 158272, 158275, 158276, 158282, 158283, 158284, 158286, 158335, 158379, 158387, 158391, 158393]


In [18]:
"""
   Descargar y procesar las OT faltantes y añadirlas al Delta Lake
"""
new_data_frames = []
for ot in new_ids_to_process:
  json_ot = CurrentCollection.find_one({"id_ot": ot})
  if not json_ot:
    print(f"No se pudo encontrar la OT con id_ot '{ot}' en MongoDB. Saltando.")
    continue
                
  obj_ot = OrdenTrabajo.GestionOt.from_dict(json_ot)
  new_data_frames.append(Actividades.ConvertirOT_a_ActividadesCSV(obj_ot))

/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarni

In [19]:
"""
   Padnas Dataframe con las OT para ser añadidas al DF
""" 
new_df = pd.concat(new_data_frames, ignore_index=True)
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 632 entries, 0 to 631
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Item           632 non-null    int64 
 1   Cuenta         632 non-null    object
 2   Evento         632 non-null    object
 3   Actividad      632 non-null    object
 4   Alimentador    632 non-null    object
 5   Primario       632 non-null    object
 6   Desconexion    632 non-null    object
 7   SIG            632 non-null    object
 8   Tipo           632 non-null    object
 9   Materiales     632 non-null    object
 10  Cuadrilla      632 non-null    object
 11  Dia            632 non-null    object
 12  Fecha          632 non-null    object
 13  InicioEvento   632 non-null    object
 14  FinEvento      632 non-null    object
 15  Duracion       632 non-null    int64 
 16  Responsable    632 non-null    object
 17  Colaboradores  632 non-null    int64 
 18  HorasExtra     632 non-null   

In [ ]:
try:
  write_deltalake(DELTA_PATH, new_df, mode='append')
  print(f" [ EXITO ] DELTA LAKE Se han añadido {len(new_df)} filas a la tabla Delta en '{DELTA_PATH}'.")
except Exception as e:
  print(f"Fallo al escribir en la tabla Delta: {e}")

### Comparando las dos Delta-Lake

In [20]:
from deltalake import DeltaTable
import pyarrow as pa
import pandas as pd

# Define the path to your Delta table
# (Using the path from your concat_ot.py script)
table_path = "./test/deltalake_2025"



In [21]:

# 2. Get the Delta-specific schema object
delta_schema = dt.schema()

# 3. Convert it to a PyArrow schema for easy inspection
pyarrow_schema = delta_schema.to_arrow()

# 4. Extract and print the column names and types
print(f"Schema for Delta table at: '{DELTA_PATH}'")
print("-" * 40)

# Get lists of names and types
column_names = pyarrow_schema.names
column_types = pyarrow_schema.types

# Print them in a readable format
for name, dtype in zip(column_names, column_types):
    print(f"Column: {name:<20} | Type: {dtype}")

print("-" * 40)

# You can also access a specific field
print(f"Details for the first field: {pyarrow_schema.field(0)}")



Schema for Delta table at: '/home/vlad/delta_v23'
----------------------------------------
Column: Item                 | Type: arro3.core.DataType<Int64>

Column: Cuenta               | Type: arro3.core.DataType<Utf8>

Column: Evento               | Type: arro3.core.DataType<Utf8>

Column: Actividad            | Type: arro3.core.DataType<Utf8>

Column: Alimentador          | Type: arro3.core.DataType<Utf8>

Column: Primario             | Type: arro3.core.DataType<Utf8>

Column: Desconexion          | Type: arro3.core.DataType<Utf8>

Column: SIG                  | Type: arro3.core.DataType<Utf8>

Column: Tipo                 | Type: arro3.core.DataType<Utf8>

Column: Materiales           | Type: arro3.core.DataType<Utf8>

Column: Cuadrilla            | Type: arro3.core.DataType<Utf8>

Column: Dia                  | Type: arro3.core.DataType<Utf8>

Column: Fecha                | Type: arro3.core.DataType<Timestamp(Microsecond, Some("UTC"))>

Column: InicioEvento         | Type: arro3.co

In [22]:
dt_test = DeltaTable(table_path)

# 2. Get the Delta-specific schema object
delta_schema = dt_test.schema()

# 3. Convert it to a PyArrow schema for easy inspection
pyarrow_schema = delta_schema.to_arrow()

# 4. Extract and print the column names and types
print(f"Schema for Delta table at: '{DELTA_PATH}'")
print("-" * 40)

# Get lists of names and types
column_names = pyarrow_schema.names
column_types = pyarrow_schema.types

# Print them in a readable format
for name, dtype in zip(column_names, column_types):
    print(f"Column: {name:<20} | Type: {dtype}")

print("-" * 40)

# You can also access a specific field
print(f"Details for the first field: {pyarrow_schema.field(0)}")



Schema for Delta table at: '/home/vlad/delta_v23'
----------------------------------------
Column: Item                 | Type: arro3.core.DataType<Int64>

Column: Cuenta               | Type: arro3.core.DataType<Utf8>

Column: Evento               | Type: arro3.core.DataType<Utf8>

Column: Actividad            | Type: arro3.core.DataType<Utf8>

Column: Alimentador          | Type: arro3.core.DataType<Utf8>

Column: Primario             | Type: arro3.core.DataType<Utf8>

Column: Desconexion          | Type: arro3.core.DataType<Utf8>

Column: SIG                  | Type: arro3.core.DataType<Utf8>

Column: Tipo                 | Type: arro3.core.DataType<Utf8>

Column: Materiales           | Type: arro3.core.DataType<Utf8>

Column: Cuadrilla            | Type: arro3.core.DataType<Utf8>

Column: Dia                  | Type: arro3.core.DataType<Utf8>

Column: Fecha                | Type: arro3.core.DataType<Utf8>

Column: InicioEvento         | Type: arro3.core.DataType<Utf8>

Column: FinE